In [ ]:
import os
from getpass import getpass

# Secure input (not visible)
os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API Key: ")
os.environ["OPENROUTER_API_KEY"] = getpass("Enter OpenRouter API Key: ")
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Gemini API Key: ··········
Enter OpenRouter API Key: ··········
Enter Groq API Key: ··········


In [ ]:
# .env
# API keys should be stored securely in environment variables, not in code

In [ ]:
!pip install pytesseract opencv-python pillow requests
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Task -1: The Timeline Recreation

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

Saving gpt papers sequence.png to gpt papers sequence (1).png


## OCR Fallback Mechanism
### Gemini 3 Flash Preview → Qwen 3.6 Plus Preview (OpenRouter) → Tesseract OCR

In [ ]:
import base64
import requests
import pytesseract
import cv2

# --- Utility: encode image ---
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
def gemini_ocr(image_path):
    try:
        print("Trying Gemini OCR...")

        base64_img = encode_image(image_path)

        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent?key={os.environ['GEMINI_API_KEY']}"

        payload = {
            "contents": [{
                "parts": [
                    {"text":"""
Extract ALL text from this image.

IMPORTANT:
- Do NOT summarize
- Do NOT skip small text
- Include EVERY year, label, and description
- Preserve order

Return plain text only.
"""},
                    {
                        "inline_data": {
                            "mime_type": "image/png",
                            "data": base64_img
                        }
                    }
                ]
            }]
        }

        res = requests.post(url, json=payload)

        if res.status_code == 200:
            text = res.json()["candidates"][0]["content"]["parts"][0]["text"]
            print("Gemini OCR Success")
            return text

        return None

    except:
        return None

In [ ]:
def qwen_ocr(image_path):
    try:
        print("Trying Qwen OCR...")

        base64_img = encode_image(image_path)

        url = "https://openrouter.ai/api/v1/chat/completions"

        headers = {
            "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"
        }

        payload = {
            "model": "qwen/qwen3.6-plus-preview",
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text":"""
Extract complete text from image.

Rules:
- Include ALL timeline entries
- Do not miss any year (e.g., 2014, 2017, etc.)
- Do not summarize

Return full raw text.
"""},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_img}"
                            }
                        }
                    ]
                }
            ]
        }

        res = requests.post(url, headers=headers, json=payload)

        if res.status_code == 200:
            text = res.json()["choices"][0]["message"]["content"]
            print("Qwen OCR Success")
            return text

        return None

    except:
        return None

In [ ]:
import cv2
import pytesseract

def preprocess(img_path):
    img = cv2.imread(img_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Increase contrast
    gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=0)

    # Adaptive threshold (better than fixed)
    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

    return thresh

In [ ]:
def tesseract_ocr(image_path):
    print("Using Tesseract OCR (multi-pass)...")

    img = preprocess(image_path)

    texts = []

    # Pass 1: default
    texts.append(pytesseract.image_to_string(img, config='--psm 6'))

    # Pass 2: sparse text (important for timelines)
    texts.append(pytesseract.image_to_string(img, config='--psm 11'))

    # Pass 3: single block
    texts.append(pytesseract.image_to_string(img, config='--psm 4'))

    # Combine results
    final_text = "\n".join(texts)

    return final_text

In [ ]:
def extract_text(image_path):

    # 1️⃣ Gemini
    text = gemini_ocr(image_path)
    if text and len(text.strip()) > 50:
        return text

    # 2️⃣ Qwen
    text = qwen_ocr(image_path)
    if text and len(text.strip()) > 50:
        return text

    # 3️⃣ Tesseract
    return tesseract_ocr(image_path)


ocr_text = extract_text(image_path)

print("\n--- FINAL OCR OUTPUT ---\n")
print(ocr_text)

Trying Gemini OCR...
Gemini OCR Success

--- FINAL OCR OUTPUT ---

Simplify
Attention is all you need (Vaswani et.al).

Just ask
Language models are few shot learners
(Brown et.al.)

RLHF
Training language models to follow instructions.

Use tools
Toolformer: Language Models can teach themselves to use tools (Schick, Yu)

14 15 16 17 18 19 20 21 22 23 24

Birth of attention
Neural machine translation by jointly learning to align and translate

Generalize
Improving language understanding by Generative pre-training (Radford et.al)

Scale to scenarios
LORA: Low-Rank Adaptation of Large Language Models (Hu, Shen, et.al.)


In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\.\-\(\)]', '', text)
    text = re.sub(r'\s+', ' ', text)

    # fix common OCR errors
    text = text.replace("lora", "LoRA")
    text = text.replace("transformer", "Transformer")

    return text.strip()

clean_text = normalize_text(ocr_text)

print("\nCLEAN TEXT:\n", clean_text)


CLEAN TEXT:
 simplify attention is all you need (vaswani et.al). just ask language models are few shot learners (brown et.al.) rlhf training language models to follow instructions. use tools toolformer language models can teach themselves to use tools (schick yu) 14 15 16 17 18 19 20 21 22 23 24 birth of attention neural machine translation by jointly learning to align and translate generalize improving language understanding by generative pre-training (radford et.al) scale to scenarios LoRA low-rank adaptation of large language models (hu shen et.al.)


In [ ]:
def enrich_with_llm(text):
    return run_llm_pipeline(prompt)  # using updated prompt above

In [ ]:
import requests

def call_llm(api_url, headers, payload):
    response = requests.post(api_url, headers=headers, json=payload)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"API call failed with status code: {response.status_code}")
        print(f"Response text: {response.text}")
        return None

> If we want the missing timelines also then we can add simple line in the prompt like infer the missing entries if pattern suggests (missing years).

In [ ]:
def gemini_llm(text):
    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent?key={os.environ['GEMINI_API_KEY']}"

        payload = {
            "contents": [{
                "parts": [{
                    "text": f"""
You are given OCR extracted text from a timeline.

Task:
1. Extract structured timeline
2. Fix OCR mistakes
3. Ensure timeline continuity

Extract timeline and ENRICH it.

For each entry include:
- year
- title
- description (cleaned)
- impact (why important)
- innovation (what changed)
- paper_link (if known, else null)
- category (Attention / GPT / RLHF / Tooling)

Return STRICT JSON:
[
  {{
    "year": int,
    "title": "",
    "description": "",
    "impact": "",
    "innovation": "",
    "paper_link": "",
    "category": ""
  }}
]

Text:
{text}
"""
                }]
            }]
        }

        res = call_llm(url, {"Content-Type": "application/json"}, payload)

        return res["candidates"][0]["content"]["parts"][0]["text"]

    except Exception as e:
        print(f"Gemini LLM error: {e}")
        return None

In [ ]:
def openrouter_llm(text):
    try:
        url = "https://openrouter.ai/api/v1/chat/completions"

        headers = {
            "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"
        }

        payload = {
            "model": "google/gemma-4-31b-it:free",
            "messages": [{
                "role": "user",
                "content": f"""
You are given OCR extracted text from a timeline.

Task:
1. Extract structured timeline
2. Fix OCR mistakes
3. Infer missing entries if pattern suggests gaps (e.g., missing years)
4. Ensure timeline continuity

Extract timeline and ENRICH it.

For each entry include:
- year
- title
- description (cleaned)
- impact (why important)
- innovation (what changed)
- paper_link (if known, else null)
- category (Attention / GPT / RLHF / Tooling)

Return STRICT JSON:
[
  {{
    "year": int,
    "title": "",
    "description": "",
    "impact": "",
    "innovation": "",
    "paper_link": "",
    "category": ""
  }}
]

Text:
{text}
"""
            }]
        }

        res = call_llm(url, headers, payload)
        return res["choices"][0]["message"]["content"]

    except Exception as e:
        print(f"OpenRouter LLM error: {e}")
        return None

In [ ]:
def grok_llm(text):
    try:
        url = "https://api.x.ai/v1/chat/completions"

        headers = {
            "Authorization": f"Bearer {os.environ['GROK_API_KEY']}"
        }

        payload = {
            "model": "openai/gpt-oss-120b",
            "messages": [{
                "role": "user",
                "content": f"""
Extract timeline and ENRICH it.

For each entry include:
- year
- title
- description (cleaned)
- impact (why important)
- innovation (what changed)
- paper_link (if known, else null)
- category (Attention / GPT / RLHF / Tooling)

Return STRICT JSON:
[
  {{
    "year": int,
    "title": "",
    "description": "",
    "impact": "",
    "innovation": "",
    "paper_link": "",
    "category": ""
  }}
]

Text:
{text}
"""
            }]
        }

        res = call_llm(url, headers, payload)
        return res["choices"][0]["message"]["content"]

    except Exception as e:
        print(f"Grok LLM error: {e}")
        return None

In [ ]:
def run_llm_pipeline(text):
    print("Trying Gemini...")
    out = gemini_llm(text)
    if out:
        print("Gemini Success")
        return out

    print("Trying OpenRouter...")
    out = openrouter_llm(text)
    if out:
        print("OpenRouter Success")
        return out

    print("Trying Grok...")
    out = grok_llm(text)
    if out:
        print("Grok Success")
        return out

    raise Exception("All LLMs failed")

llm_output = run_llm_pipeline(clean_text)

print("\nLLM OUTPUT:\n", llm_output)

Trying Gemini...
Gemini Success

LLM OUTPUT:
 ```json
[
  {
    "year": 2014,
    "title": "Neural Machine Translation by Jointly Learning to Align and Translate",
    "description": "Introduction of the first attention mechanism to overcome the bottleneck of fixed-length vectors in encoder-decoder architectures.",
    "impact": "Revolutionized machine translation by allowing the model to focus on specific parts of the input sequence, laying the groundwork for all future attention-based models.",
    "innovation": "Introduced the 'Alignment' mechanism (additive attention), allowing models to search for relevant parts of a source sentence.",
    "paper_link": "https://arxiv.org/abs/1409.0473",
    "category": "Attention"
  },
  {
    "year": 2017,
    "title": "Attention Is All You Need",
    "description": "Introduction of the Transformer architecture, which entirely replaces recurrence and convolutions with self-attention mechanisms.",
    "impact": "Became the foundational architectu

In [ ]:
import json
import re

def safe_parse(text):
    try:
        # Clean the LLM output if it contains markdown code blocks
        clean_json = text.strip()
        if "```json" in clean_json:
            clean_json = clean_json.split("```json")[1].split("```")[0].strip()
        elif "```" in clean_json:
            clean_json = clean_json.split("```")[1].split("```")[0].strip()

        return json.loads(clean_json)
    except Exception as e:
        print(f"Initial parse failed: {e}")
        # Attempt a more aggressive regex-based extraction if needed
        return []

# Re-parse to ensure we get impact, innovation, paper_link, and category
timeline_data = safe_parse(llm_output)

print(f"Successfully parsed {len(timeline_data)} entries with detailed fields.")
if timeline_data:
    display(timeline_data[0])

Successfully parsed 7 entries with detailed fields.


{'year': 2014,
 'title': 'Neural Machine Translation by Jointly Learning to Align and Translate',
 'description': 'Introduction of the first attention mechanism to overcome the bottleneck of fixed-length vectors in encoder-decoder architectures.',
 'impact': 'Revolutionized machine translation by allowing the model to focus on specific parts of the input sequence, laying the groundwork for all future attention-based models.',
 'innovation': "Introduced the 'Alignment' mechanism (additive attention), allowing models to search for relevant parts of a source sentence.",
 'paper_link': 'https://arxiv.org/abs/1409.0473',
 'category': 'Attention'}

In [ ]:
def validate_timeline(data):
    years = [item["year"] for item in data]

    expected_range = list(range(min(years), max(years)+1))

    missing = set(expected_range) - set(years)

    if missing:
        print("⚠ Missing years detected:", missing)

    return data

validated_timeline = validate_timeline(timeline_data)

⚠ Missing years detected: {2016, 2019, 2015}


In [ ]:
from IPython.display import HTML
import json

if 'timeline_data' in globals():
    js_data = json.dumps(timeline_data)

    html_template = f"""
    <script src='https://cdn.tailwindcss.com'></script>
    <style>
      @keyframes pulse-slow {{ 0%, 100% {{ opacity: 0.3; }} 50% {{ opacity: 0.6; }} }}
      .glass-effect {{ backdrop-filter: blur(12px); transition: all 0.3s ease; }}
      .dark .glass-effect {{ background: rgba(15, 23, 42, 0.8); border-color: rgba(51, 65, 85, 0.5); }}
      .light .glass-effect {{ background: rgba(255, 255, 255, 0.8); border-color: rgba(226, 232, 240, 0.8); }}
      .timeline-path {{ background: linear-gradient(to bottom, transparent, #06b6d4 10%, #d946ef 90%, transparent); }}
      .highlight-zone {{ transition: all 0.4s ease; }}
      .group:hover .highlight-zone {{ background: rgba(6, 182, 212, 0.05); border-radius: 1rem; padding: 0.5rem; box-shadow: 0 0 20px rgba(6, 182, 212, 0.1); }}
      body {{ transition: background-color 0.3s ease, color 0.3s ease; }}
    </style>

    <div id='themeWrapper' class='dark'>
      <div class='p-8 bg-[#020617] dark:bg-[#020617] light:bg-slate-50 text-slate-200 dark:text-slate-200 light:text-slate-800 min-h-screen font-sans selection:bg-cyan-500/30 relative overflow-hidden' id='mainContainer'>
          <div class='absolute inset-0 z-0 pointer-events-none'>
              <div class='absolute top-[-10%] left-[-10%] w-[40%] h-[40%] bg-cyan-500/10 blur-[120px] rounded-full animate-pulse'></div>
              <div class='absolute bottom-[-10%] right-[-10%] w-[40%] h-[40%] bg-fuchsia-500/10 blur-[120px] rounded-full animate-pulse' style='animation-delay: 2s;'></div>
          </div>

          <div class='max-w-6xl mx-auto relative z-10'>
              <div class='flex justify-end mb-4'>
                  <button id='themeToggle' class='glass-effect border p-3 rounded-2xl flex items-center gap-2 hover:scale-105 transition-all'>
                      <span id='themeIcon'>🌙</span> <span class='text-xs font-bold uppercase tracking-widest'>Toggle Theme</span>
                  </button>
              </div>

              <header class='text-center mb-16'>
                  <h1 class='text-7xl font-black mb-4 bg-clip-text text-transparent bg-gradient-to-r from-cyan-400 via-blue-500 to-fuchsia-500 tracking-tighter'>
                      GPT Evolution
                  </h1>
                  <p class='text-slate-400 dark:text-slate-400 light:text-slate-500 text-xl font-light tracking-wide mb-8'>Tracing the Architectural Ancestry of Modern AI</p>
                  <div id='statsBar' class='flex justify-center gap-8 py-4 border-y border-slate-800/50 dark:border-slate-800/50 light:border-slate-200 mb-12'></div>
              </header>

              <div class='sticky top-6 z-50 mb-12'>
                  <div class='glass-effect border p-3 rounded-3xl shadow-2xl flex flex-col md:flex-row gap-3'>
                      <input type='text' id='searchBar' placeholder='Search innovations...' class='flex-1 bg-slate-900/80 dark:bg-slate-900/80 light:bg-white/80 border border-slate-700 dark:border-slate-700 light:border-slate-200 p-4 rounded-2xl outline-none focus:ring-2 focus:ring-cyan-500/50 transition-all'>
                      <select id='categorySelect' class='bg-slate-900/80 dark:bg-slate-900/80 light:bg-white/80 border border-slate-700 dark:border-slate-700 light:border-slate-200 p-4 rounded-2xl outline-none cursor-pointer'>
                          <option value='all'>All Categories</option>
                          <option value='Attention'>Attention</option>
                          <option value='RLHF'>RLHF</option>
                          <option value='GPT'>GPT</option>
                          <option value='Tooling'>Tooling</option>
                      </select>
                  </div>
              </div>

              <div class='relative'>
                  <div class='absolute left-4 md:left-1/2 top-0 bottom-0 w-1 timeline-path md:-translate-x-1/2 opacity-30'></div>
                  <div id='timelineRoot' class='space-y-24'></div>
              </div>
          </div>
      </div>
    </div>

    <script>
    (function() {{
        const data = {js_data};
        const root = document.getElementById('timelineRoot');
        const statsBar = document.getElementById('statsBar');
        const themeWrapper = document.getElementById('themeWrapper');
        const mainContainer = document.getElementById('mainContainer');
        const themeToggle = document.getElementById('themeToggle');
        const themeIcon = document.getElementById('themeIcon');

        const colorMap = {{ 'Attention': 'cyan', 'GPT': 'emerald', 'RLHF': 'amber', 'Tooling': 'fuchsia' }};

        themeToggle.addEventListener('click', () => {{
            const isDark = themeWrapper.classList.contains('dark');
            themeWrapper.classList.toggle('dark', !isDark);
            themeWrapper.classList.toggle('light', isDark);
            themeIcon.innerText = isDark ? '☀️' : '🌙';
            mainContainer.style.backgroundColor = isDark ? '#f8fafc' : '#020617';
            mainContainer.style.color = isDark ? '#1e293b' : '#f1f5f9';
        }});

        function updateStats(items) {{
            const cats = [...new Set(items.map(i => i.category))];
            statsBar.innerHTML = `
                <div class='text-center'><span class='block text-2xl font-bold'>${{items.length}}</span><span class='text-[10px] uppercase opacity-50'>Milestones</span></div>
                <div class='w-px h-8 bg-slate-800 dark:bg-slate-800 light:bg-slate-200'></div>
                <div class='text-center'><span class='block text-2xl font-bold'>${{cats.length}}</span><span class='text-[10px] uppercase opacity-50'>Eras</span></div>
                <div class='w-px h-8 bg-slate-800 dark:bg-slate-800 light:bg-slate-200'></div>
                <div class='text-center'><span class='block text-2xl font-bold'>${{items.length > 0 ? items[items.length-1].year - items[0].year : 0}}+</span><span class='text-[10px] uppercase opacity-50'>Year Span</span></div>
            `;
        }}

        function render(items) {{
            root.innerHTML = '';
            if (items.length === 0) return;
            updateStats(items);
            items.forEach((item, i) => {{
                const color = colorMap[item.category] || 'slate';
                const isEven = i % 2 === 0;
                const entry = document.createElement('div');
                entry.className = "relative flex flex-col md:flex-row items-center gap-12 " + (isEven ? "md:flex-row-reverse" : "");
                entry.innerHTML = `
                    <div class='absolute left-4 md:left-1/2 w-6 h-6 rounded-full bg-${{color}}-500 shadow-[0_0_30px_rgba(6,182,212,0.6)] z-20 md:-translate-x-1/2 border-4 border-[#020617] dark:border-[#020617] light:border-white'></div>
                    <div class='flex-1 w-full ml-12 md:ml-0'>
                        <div class='glass-effect border p-8 rounded-[2.5rem] hover:scale-[1.01] hover:border-${{color}}-500/50 transition-all duration-500 shadow-2xl group'>
                            <div class='flex justify-between items-center mb-6'>
                                <span class='text-6xl font-black opacity-20 group-hover:text-${{color}}-500 transition-colors'>${{item.year}}</span>
                                <span class='px-4 py-1 rounded-full bg-${{color}}-500/10 text-${{color}}-400 text-xs font-bold uppercase'>${{item.category}}</span>
                            </div>
                            <h3 class='text-3xl font-bold mb-4'>${{item.title}}</h3>
                            <p class='opacity-70 text-lg mb-6'>${{item.description}}</p>
                            <div class='space-y-4 pt-4 border-t border-slate-800 dark:border-slate-800 light:border-slate-200 highlight-zone'>
                                <p class='text-sm leading-relaxed'><strong class='text-${{color}}-500'>Impact:</strong> ${{item.impact || 'N/A'}}</p>
                                <p class='text-sm leading-relaxed'><strong class='text-${{color}}-500'>Innovation:</strong> ${{item.innovation || 'N/A'}}</p>
                                ${{item.paper_link ? `<a href="${{item.paper_link}}" target="_blank" class="inline-flex items-center mt-4 px-6 py-2 rounded-xl bg-${{color}}-500/10 border border-${{color}}-500/20 text-${{color}}-500 text-sm hover:bg-${{color}}-500/20 transition-all font-bold">Read Research Paper <span class='ml-2 text-lg'>→</span></a>` : ''}}
                            </div>
                        </div>
                    </div>
                    <div class='hidden md:block flex-1'></div>
                `;
                root.appendChild(entry);
            }});
        }}

        document.getElementById('searchBar').addEventListener('input', (e) => {{
            const q = e.target.value.toLowerCase();
            render(data.filter(d => d.title.toLowerCase().includes(q) || d.description.toLowerCase().includes(q)));
        }});

        document.getElementById('categorySelect').addEventListener('change', (e) => {{
            const cat = e.target.value;
            render(cat === 'all' ? data : data.filter(d => d.category === cat));
        }});

        render(data);
    }})();
    </script>
    """
    display(HTML(html_template))
else:
    print("Error: timeline_data variable not found.")


In [ ]:
import os

# We use the html_template variable created in the previous cell
if 'html_template' in globals():
    with open('index.html', 'w') as f:
        # We strip the Jupyter-specific display logic and just save the string
        f.write(html_template)

    print("Successfully created 'index.html'!")
    from google.colab import files
    files.download('index.html')
else:
    print("Error: Could not find the dashboard template. Please run the visualization cell first.")

Successfully created 'index.html'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from google.colab import files

def export_timeline(data, filename='enriched_timeline.json'):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)
    print(f'Timeline data saved to {filename}')
    files.download(filename)

# Uncomment the line below to download the JSON file
# export_timeline(timeline_data)

In [ ]:
# from IPython.display import HTML
# import json

# color_map = {
#     "Attention": "bg-blue-600",
#     "GPT": "bg-green-600",
#     "RLHF": "bg-yellow-600",
#     "Tooling": "bg-purple-600",
#     "Unknown": "bg-gray-700"
# }

# js_script = """
# <script>
#     function filterTimeline() {
#         var selectedCategory = document.getElementById('categoryFilter').value;
#         var searchTerm = document.getElementById('timelineSearch').value.toLowerCase();
#         var cards = document.querySelectorAll('.timeline-card');

#         cards.forEach(function(card) {
#             var cardCategory = card.getAttribute('data-category');
#             var cardSearchText = card.getAttribute('data-search-text');

#             var matchesCategory = (selectedCategory === 'all' || cardCategory === selectedCategory);
#             var matchesSearch = cardSearchText.includes(searchTerm);

#             if (matchesCategory && matchesSearch) {
#                 card.style.display = 'block';
#             } else {
#                 card.style.display = 'none';
#             }
#         });
#     }
# </script>
# """

# # Build the event cards using .get() to avoid KeyErrors
# event_html = ""
# for item in timeline_data:
#     cat = item.get("category", "Unknown")
#     color = color_map.get(cat, color_map["Unknown"])
#     title = item.get("title", "N/A")
#     year = item.get("year", "N/A")
#     desc = item.get("description", "N/A")
#     impact = item.get("impact", "N/A")
#     innovation = item.get("innovation", "N/A")
#     link = item.get("paper_link", "")

#     # Pre-calculate search text
#     search_text = f"{title} {desc} {year} {impact} {innovation} {cat}".lower()

#     event_html += f"""
#     <div class='mb-10 ml-6 group cursor-pointer timeline-card'
#          data-category='{cat}'
#          data-search-text='{search_text}'>
#       <div class='absolute w-4 h-4 {color} rounded-full -left-2 top-2 border-2 border-white'></div>
#       <div class='p-5 rounded-lg shadow-xl {color} border border-white/10 hover:scale-[1.02] transition-transform duration-200'>
#         <div class='flex justify-between items-start mb-2'>
#           <h2 class='text-xl font-bold'>{title}</h2>
#           <span class='bg-black/30 px-2 py-1 rounded text-xs'>{year}</span>
#         </div>
#         <p class='text-sm text-white/90 leading-relaxed mb-4'>{desc}</p>
#         <div class='space-y-2 border-t border-white/20 pt-3 text-sm'>
#           <p><strong>Impact:</strong> {impact}</p>
#           <p><strong>Innovation:</strong> {innovation}</p>
#           {f'<a href="{link}" target="_blank" class="inline-block mt-2 text-blue-200 hover:text-white underline font-medium">Read Paper →</a>' if link else ''}
#         </div>
#       </div>
#     </div>
#     """

# html = f"""
# <script src="https://cdn.tailwindcss.com"></script>
# <div class="p-8 bg-gray-900 text-white min-h-screen">
#   <h1 class="text-4xl font-extrabold mb-4 text-center bg-clip-text text-transparent bg-gradient-to-r from-blue-400 to-purple-400">GPT Evolution Timeline</h1>
#   <p class="text-center text-gray-400 mb-10">Tracing the path from Attention mechanisms to Autonomous Tool Use</p>

#   <div class="mb-8 flex flex-wrap gap-4 justify-center sticky top-4 z-10">
#       <select id="categoryFilter" onchange="filterTimeline()" class="p-2 rounded-lg bg-gray-800 text-white border border-gray-700 shadow-lg">
#           <option value="all">All Categories</option>
#           <option value="Attention">Attention</option>
#           <option value="GPT">GPT</option>
#           <option value="RLHF">RLHF</option>
#           <option value="Tooling">Tooling</option>
#       </select>
#       <input type="text" id="timelineSearch" oninput="filterTimeline()" placeholder="Search papers, years, or tech..." class="p-2 w-64 rounded-lg bg-gray-800 text-white border border-gray-700 shadow-lg"/>
#   </div>

#   <div id="timeline-events" class="relative border-l-2 border-dashed border-gray-700 ml-10 max-w-4xl mx-auto">
#     {event_html}
#   </div>
# </div>
# {js_script}
# """

# display(HTML(html))

# Task-2: Learning Material Creation

In [ ]:
# Step 0: Install all dependencies
!pip install -q pymupdf pdf2image pytesseract Pillow langchain langchain-google-genai openai requests python-dotenv google-generativeai

# System dependency for Tesseract
!apt-get install -q tesseract-ocr tesseract-ocr-eng

# For PDF→image conversion
!apt-get install -q poppler-utils

print("✅ All dependencies installed")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ All dependencies installed


In [ ]:
# Step 1: Convert each PDF page to a high-res image
import fitz  # PyMuPDF
import os
from PIL import Image

def pdf_to_images(pdf_path: str, output_dir: str = "page_images", dpi: int = 200) -> list[dict]:
    """
    Converts each PDF page to a PNG image.
    Also extracts embedded images per page for later injection.
    Returns a list of dicts: {page_num, image_path, embedded_images[]}
    """
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]

        # Render full page as image
        mat = fitz.Matrix(dpi / 72, dpi / 72)  # scale matrix
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        page_img_path = f"{output_dir}/page_{page_num + 1:03d}.png"
        pix.save(page_img_path)

        # Extract embedded images within the page
        embedded = []
        img_list = page.get_images(full=True)
        for img_idx, img_info in enumerate(img_list):
            xref = img_info[0]
            base_image = doc.extract_image(xref)
            img_bytes = base_image["image"]
            img_ext = base_image["ext"]
            emb_path = f"{output_dir}/page_{page_num+1:03d}_img_{img_idx+1}.{img_ext}"
            with open(emb_path, "wb") as f:
                f.write(img_bytes)
            embedded.append(emb_path)

        pages.append({
            "page_num": page_num + 1,
            "image_path": page_img_path,
            "embedded_images": embedded
        })
        print(f"  Page {page_num + 1}: rendered + {len(embedded)} embedded image(s) extracted")

    doc.close()
    print(f"\n✅ {len(pages)} pages converted → '{output_dir}/'")
    return pages

# Run it
PDF_PATH = "/content/drive/MyDrive/Sarala Labs/Bootcamp2 pdf - emw chapter.pdf"  # update path as needed
pages_data = pdf_to_images(PDF_PATH)

  Page 1: rendered + 1 embedded image(s) extracted
  Page 2: rendered + 1 embedded image(s) extracted
  Page 3: rendered + 1 embedded image(s) extracted
  Page 4: rendered + 1 embedded image(s) extracted
  Page 5: rendered + 1 embedded image(s) extracted
  Page 6: rendered + 1 embedded image(s) extracted
  Page 7: rendered + 1 embedded image(s) extracted
  Page 8: rendered + 1 embedded image(s) extracted
  Page 9: rendered + 1 embedded image(s) extracted
  Page 10: rendered + 1 embedded image(s) extracted
  Page 11: rendered + 1 embedded image(s) extracted
  Page 12: rendered + 1 embedded image(s) extracted
  Page 13: rendered + 1 embedded image(s) extracted
  Page 14: rendered + 1 embedded image(s) extracted

✅ 14 pages converted → 'page_images/'


In [ ]:
# Step 2: OCR — one image at a time, raw REST API (no genai SDK), 30s timeout, saves after each page

import pytesseract
import base64, requests, os, json, time
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

# ── CONFIG ────────────────────────────────────────────────────────────────────
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

TIMEOUT_SECS       = 30                          # hard timeout per model attempt
MIN_TEXT_LEN       = 30                          # minimum chars to accept a result
RESULTS_FILE       = "ocr_results.json"          # incremental save path

OCR_PROMPT = (
    "You are an expert OCR engine. Extract ALL text from this image EXACTLY as it appears. "
    "Rules: "
    "Math equations → LaTeX: inline $...$, block $$...$$. "
    "Preserve section headings, figure captions, table content. "
    "Do NOT summarize or skip anything. "
    "Output raw extracted text only."
)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def call_with_timeout(fn, timeout: int = TIMEOUT_SECS):
    """
    Runs fn() with a hard timeout.
    Returns (result, elapsed, timed_out).
    """
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(fn)
        try:
            result  = future.result(timeout=timeout)
            elapsed = round(time.time() - t0, 1)
            return result, elapsed, False
        except FuturesTimeout:
            return None, timeout, True
        except Exception as e:
            return None, round(time.time() - t0, 1), False

def load_existing_results() -> dict:
    """Load already-processed pages so we can resume if interrupted."""
    if Path(RESULTS_FILE).exists():
        with open(RESULTS_FILE) as f:
            data = json.load(f)
        print(f"♻️  Resuming — {len(data)} pages already processed")
        return data            # {str(page_num): result_dict}
    return {}

def save_result(page_num: int, result: dict, all_results: dict):
    """Save immediately after each page — crash-safe."""
    all_results[str(page_num)] = result
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

# ── MODEL 1: Gemini via raw REST API ─────────────────────────────────────────
def _gemini_ocr(image_path: str) -> str:
    """
    Calls Gemini 2.0 Flash directly via REST — no SDK dependency.
    Raises on any error so call_with_timeout can catch it.
    """
    b64  = image_to_base64(image_path)
    url  = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}"
    )
    payload = {
        "contents": [{
            "parts": [
                {"text": OCR_PROMPT},
                {
                    "inline_data": {
                        "mime_type": "image/png",
                        "data": b64
                    }
                }
            ]
        }],
        "generationConfig": {
            "temperature": 0             # deterministic — best for OCR
        }
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECS)
    resp.raise_for_status()
    data = resp.json()
    return data["candidates"][0]["content"]["parts"][0]["text"].strip()


# ── MODEL 2: Qwen VL via OpenRouter REST API ──────────────────────────────────
def _qwen_ocr(image_path: str) -> str:
    """
    Calls Qwen 2.5 VL via OpenRouter REST.
    Raises on any error so call_with_timeout can catch it.
    """
    b64  = image_to_base64(image_path)
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://colab.research.google.com",   # required by OpenRouter
        },
        json={
            "model": "qwen/qwen2.5-vl-72b-instruct:free",
            "messages": [{
                "role": "user",
                "content": [
                    {
                        "type":      "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"}
                    },
                    {
                        "type": "text",
                        "text": OCR_PROMPT
                    }
                ]
            }],
            "temperature": 0
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()


# ── MODEL 3: Tesseract (local fallback — always fast) ─────────────────────────
def _tesseract_ocr(image_path: str) -> str:
    """
    Local Tesseract — no network, almost never times out.
    Raises on any error so call_with_timeout can catch it.
    """
    img  = Image.open(image_path)
    text = pytesseract.image_to_string(img, config="--psm 6 --oem 3")
    if not text.strip():
        raise ValueError("Tesseract returned empty text")
    return text.strip()


# ── PER-PAGE OCR WITH FALLBACK CHAIN ─────────────────────────────────────────
MODELS = [
    ("Gemini 2.0 Flash (REST)", _gemini_ocr),
    ("Qwen 2.5 VL (OpenRouter)", _qwen_ocr),
    ("Tesseract (local)",        _tesseract_ocr),
]

def ocr_one_page(page: dict) -> dict:
    """
    Tries each model in order with TIMEOUT_SECS timeout.
    Returns result dict as soon as one succeeds.
    Saves to disk immediately after success or total failure.
    """
    pn         = page["page_num"]
    image_path = page["image_path"]

    print(f"\n{'─'*50}")
    print(f"📄 Page {pn}  →  {Path(image_path).name}")

    for model_name, model_fn in MODELS:
        print(f"  ↳ [{model_name}] calling...", end=" ", flush=True)

        text, elapsed, timed_out = call_with_timeout(
            lambda fn=model_fn: fn(image_path),   # lambda captures current fn
            timeout=TIMEOUT_SECS
        )

        if timed_out:
            print(f"⏱️  TIMED OUT after {TIMEOUT_SECS}s — next model")
            continue

        if text and len(text.strip()) >= MIN_TEXT_LEN:
            print(f"✅ {elapsed}s — {len(text)} chars extracted")
            return {
                "page_num":        pn,
                "text":            text.strip(),
                "source":          model_name,
                "elapsed_sec":     elapsed,
                "image_path":      image_path,
                "embedded_images": page["embedded_images"],
            }

        print(f"⚠️  {elapsed}s — too little text ({len(text or '')} chars) — next model")

    # All models failed
    print(f"  ❌ All models failed for page {pn}")
    return {
        "page_num":        pn,
        "text":            "",
        "source":          "FAILED",
        "elapsed_sec":     0,
        "image_path":      image_path,
        "embedded_images": page["embedded_images"],
    }


# ── MAIN LOOP: one page at a time, saves after each ──────────────────────────
def run_ocr_sequential(pages_data: list) -> list[dict]:
    import time
    t_start     = time.time()
    all_results = load_existing_results()          # resume support
    ocr_results = []

    for page in pages_data:
        pn = page["page_num"]

        # Skip if already processed (resume mode)
        if str(pn) in all_results:
            print(f"  ⏭️  Page {pn} already done — skipping")
            ocr_results.append(all_results[str(pn)])
            continue

        result = ocr_one_page(page)

        save_result(pn, result, all_results)       # 💾 save immediately
        ocr_results.append(result)

    # ── Final summary ─────────────────────────────────────────────────────────
    total_time = round(time.time() - t_start, 1)
    success    = sum(1 for r in ocr_results if r["text"])
    sources    = {}
    for r in ocr_results:
        sources[r["source"]] = sources.get(r["source"], 0) + 1

    print(f"\n{'='*55}")
    print(f"✅  OCR complete in {total_time}s")
    print(f"    Extracted : {success} / {len(ocr_results)} pages")
    print(f"    Failed    : {len(ocr_results) - success} pages")
    print(f"    Results   : saved to '{RESULTS_FILE}'")
    print("    Sources   :")
    for src, count in sources.items():
        print(f"      {src}: {count} page(s)")
    print(f"{'='*55}")

    return sorted(ocr_results, key=lambda x: x["page_num"])


# ── RUN ───────────────────────────────────────────────────────────────────────
ocr_results = run_ocr_sequential(pages_data)

♻️  Resuming — 14 pages already processed
  ⏭️  Page 1 already done — skipping
  ⏭️  Page 2 already done — skipping
  ⏭️  Page 3 already done — skipping
  ⏭️  Page 4 already done — skipping
  ⏭️  Page 5 already done — skipping
  ⏭️  Page 6 already done — skipping
  ⏭️  Page 7 already done — skipping
  ⏭️  Page 8 already done — skipping
  ⏭️  Page 9 already done — skipping
  ⏭️  Page 10 already done — skipping
  ⏭️  Page 11 already done — skipping
  ⏭️  Page 12 already done — skipping
  ⏭️  Page 13 already done — skipping
  ⏭️  Page 14 already done — skipping

✅  OCR complete in 0.0s
    Extracted : 14 / 14 pages
    Failed    : 0 pages
    Results   : saved to 'ocr_results.json'
    Sources   :
      Gemini 3.1 Flash Lite Preview (REST): 14 page(s)


In [ ]:
# Step 3: Text preprocessing, normalization, and formula detection
import re

def preprocess_text(raw_text: str) -> dict:
    """
    Cleans OCR text and tags mathematical expressions and figure references.
    Returns cleaned text + lists of formulas and figure refs found.
    """
    text = raw_text

    # Fix common OCR artifacts
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)   # unwrap single newlines
    text = re.sub(r' {2,}', ' ', text)              # collapse multiple spaces
    text = re.sub(r'\n{3,}', '\n\n', text)          # max 2 consecutive blank lines
    text = text.replace('\x0c', '\n')               # form-feed → newline

    # Normalize unicode fractions and symbols
    replacements = {'½': '1/2', '¼': '1/4', '¾': '3/4',
                    '×': '*', '÷': '/', '−': '-', '≈': '~='}
    for old, new in replacements.items():
        text = text.replace(old, new)

    # Extract LaTeX formulas (already tagged by vision LLM)
    inline_formulas  = re.findall(r'\$(?!\$)(.+?)\$(?!\$)', text, re.DOTALL)
    display_formulas = re.findall(r'\$\$(.+?)\$\$', text, re.DOTALL)

    # Extract figure/equation references
    figure_refs = re.findall(r'(?:Fig(?:ure)?|FIGURE)\.?\s*[\d\.]+', text, re.IGNORECASE)
    eq_refs     = re.findall(r'(?:Eq(?:uation)?|Eq\.)\.?\s*[\(\d][\d\.]+[\)]?', text, re.IGNORECASE)

    return {
        "cleaned_text": text.strip(),
        "inline_formulas":  list(set(inline_formulas)),
        "display_formulas": list(set(display_formulas)),
        "figure_refs": list(set(figure_refs)),
        "equation_refs": list(set(eq_refs)),
    }

# Apply preprocessing to all OCR results
preprocessed_pages = []
for ocr in ocr_results:
    processed = preprocess_text(ocr["text"])
    preprocessed_pages.append({
        **ocr,
        **processed,
    })
    print(f"Page {ocr['page_num']:>3}: {len(processed['display_formulas'])} block formulas, "
          f"{len(processed['inline_formulas'])} inline, "
          f"{len(ocr['embedded_images'])} images")

print("\n✅ Preprocessing complete")

Page   1: 0 block formulas, 0 inline, 1 images
Page   2: 2 block formulas, 10 inline, 1 images
Page   3: 3 block formulas, 15 inline, 1 images
Page   4: 2 block formulas, 12 inline, 1 images
Page   5: 0 block formulas, 6 inline, 1 images
Page   6: 3 block formulas, 13 inline, 1 images
Page   7: 0 block formulas, 21 inline, 1 images
Page   8: 3 block formulas, 17 inline, 1 images
Page   9: 0 block formulas, 39 inline, 1 images
Page  10: 0 block formulas, 6 inline, 1 images
Page  11: 0 block formulas, 6 inline, 1 images
Page  12: 6 block formulas, 22 inline, 1 images
Page  13: 0 block formulas, 5 inline, 1 images
Page  14: 0 block formulas, 21 inline, 1 images

✅ Preprocessing complete


In [ ]:
# Step 4: LLM structures raw text into sections with page index
import json

SECTION_PROMPT_TEMPLATE = """
You are a document structuring assistant. Given the raw OCR text below (extracted from a Physics textbook chapter),
your job is to organize it into a clean structured JSON.

Output ONLY valid JSON, no markdown fences, no explanation.

JSON schema:
{{
  "chapter_title": "string",
  "sections": [
    {{
      "section_id": "string (e.g. 8.1, 8.2.1)",
      "title": "string",
      "page_range": [start_page, end_page],
      "content": "string (cleaned text for this section)",
      "key_formulas": ["LaTeX formula strings"],
      "key_concepts": ["concept1", "concept2"],
      "has_figures": true/false,
      "difficulty_hint": "foundational | intermediate | advanced"
    }}
  ],
  "summary_points": ["point1", "point2"]
}}

Raw OCR text (pages {start_page}–{end_page}):
{text}
"""

def structure_with_gemini(combined_text: str, start_page: int, end_page: int) -> dict | None:
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        prompt = SECTION_PROMPT_TEMPLATE.format(
            text=combined_text[:12000],  # token safety
            start_page=start_page, end_page=end_page
        )
        resp = model.generate_content(prompt)
        raw = resp.text.strip().lstrip("```json").rstrip("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  [Gemini structure failed]: {e}")
        return None

def structure_with_openrouter(combined_text: str, start_page: int, end_page: int,
                               model_id: str = "google/gemma-3-27b-it:free") -> dict | None:
    try:
        headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
        prompt = SECTION_PROMPT_TEMPLATE.format(
            text=combined_text[:10000], start_page=start_page, end_page=end_page
        )
        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": prompt}]
        }
        resp = requests.post("https://openrouter.ai/api/v1/chat/completions",
                             headers=headers, json=payload, timeout=90)
        raw = resp.json()["choices"][0]["message"]["content"].strip()
        raw = raw.lstrip("```json").rstrip("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  [OpenRouter structure failed ({model_id})]: {e}")
        return None

def structure_with_fallback(combined_text: str, start_page: int, end_page: int) -> dict:
    print(f"\n🗂️  Structuring pages {start_page}–{end_page}...")

    for fn, label in [
        (lambda t, s, e: structure_with_gemini(t, s, e),                             "Gemini 2.0 Flash"),
        (lambda t, s, e: structure_with_openrouter(t, s, e, "google/gemma-3-27b-it:free"), "Gemma 4 (OpenRouter)"),
        (lambda t, s, e: structure_with_openrouter(t, s, e, "x-ai/grok-3-mini-beta"),  "Grok (OpenRouter)"),
    ]:
        result = fn(combined_text, start_page, end_page)
        if result and "sections" in result:
            print(f"  ✅ Structured via {label}: {len(result['sections'])} sections")
            return result
        time.sleep(2)

    # Fallback: return minimal structure
    print("  ⚠️ All structuring failed, using flat fallback")
    return {
        "chapter_title": "Unknown Chapter",
        "sections": [{
            "section_id": "all",
            "title": "Full Content",
            "page_range": [start_page, end_page],
            "content": combined_text,
            "key_formulas": [],
            "key_concepts": [],
            "has_figures": False,
            "difficulty_hint": "intermediate"
        }],
        "summary_points": []
    }

# Combine all page text and structure
full_text = "\n\n".join(
    f"[PAGE {p['page_num']}]\n{p['cleaned_text']}"
    for p in preprocessed_pages if p['cleaned_text']
)
structured_doc = structure_with_fallback(
    full_text,
    start_page=preprocessed_pages[0]['page_num'],
    end_page=preprocessed_pages[-1]['page_num']
)

# Save structured JSON
with open("structured_doc.json", "w") as f:
    json.dump(structured_doc, f, indent=2)
print("\n✅ Structured JSON saved → structured_doc.json")
print(json.dumps(structured_doc, indent=2)[:1000], "...")


🗂️  Structuring pages 1–14...
  [Gemini structure failed]: name 'genai' is not defined
  [OpenRouter structure failed (google/gemma-3-27b-it:free)]: 'choices'
  [OpenRouter structure failed (x-ai/grok-3-mini-beta)]: 'choices'
  ⚠️ All structuring failed, using flat fallback

✅ Structured JSON saved → structured_doc.json
{
  "chapter_title": "Unknown Chapter",
  "sections": [
    {
      "section_id": "all",
      "title": "Full Content",
      "page_range": [
        1,
        14
      ],
      "content": "[PAGE 1]\nChapter Eight ELECTROMAGNETIC WAVES\n\n12089CH08\n\n8.1 INTRODUCTION In Chapter 4, we learnt that an electric current produces magnetic field and that two current-carrying wires exert a magnetic force on each other. Further, in Chapter 6, we have seen that a magnetic field changing with time gives rise to an electric field. Is the converse also true? Does an electric field changing with time give rise to a magnetic field? James Clerk Maxwell (1831-1879), argued that this 

In [ ]:
# Step 5: Build master document using LangChain Refine chain
!pip install -q langchain-google-genai langchain-core langchain-classic

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# Ensure the API key is retrieved from environment
API_KEY = os.environ.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=API_KEY,
    temperature=0.3
)

# Convert structured sections into LangChain Documents
lc_docs = []
for section in structured_doc["sections"]:
    content = (
        f"SECTION: {section['section_id']} — {section['title']}\n"
        f"Pages: {section['page_range']}\n"
        f"Difficulty: {section['difficulty_hint']}\n\n"
        f"{section['content']}\n\n"
        f"Key Formulas: {'; '.join(section['key_formulas'])}\n"
        f"Key Concepts: {', '.join(section['key_concepts'])}"
    )
    lc_docs.append(Document(page_content=content, metadata={
        "section_id": section["section_id"],
        "title": section["title"],
        "page_range": section["page_range"],
        "has_figures": section["has_figures"],
    }))

print(f"📄 {len(lc_docs)} LangChain Documents created")

# Refine chain — iteratively builds a coherent understanding
refine_initial_prompt = PromptTemplate(
    input_variables=["text"],
    template=(
        "You are a physics expert. Read this section and extract a clear, complete understanding:\n\n"
        "{text}\n\n"
        "Write a comprehensive summary preserving all key equations, concepts, and relationships."
    )
)
refine_iter_prompt = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=(
        "Here is the understanding built so far:\n{existing_answer}\n\n"
        "Now refine and extend it with this new section:\n{text}\n\n"
        "Output a single comprehensive document. Preserve all formulas and key concepts."
    )
)

refine_chain = load_summarize_chain(
    llm,
    chain_type="refine",
    question_prompt=refine_initial_prompt,
    refine_prompt=refine_iter_prompt,
    verbose=False
)

print("⚙️ Running Refine chain (this may take ~1–2 min)...")
refined_output = refine_chain.invoke(lc_docs)
master_content = refined_output["output_text"]

print(f"\n✅ Master document built ({len(master_content)} chars)")
print(master_content[:500], "...")

📄 1 LangChain Documents created
⚙️ Running Refine chain (this may take ~1–2 min)...

✅ Master document built (4255 chars)
Here's a comprehensive summary of the provided text on electromagnetic waves, covering key concepts, equations, and relationships:

**Overall Understanding:**

The chapter introduces the concept of electromagnetic waves, building upon previous knowledge of electric and magnetic fields. It begins by addressing an inconsistency in Ampere's Law, leading to the introduction of displacement current by Maxwell. This leads to Maxwell's equations, which predict the existence of electromagnetic waves. Th ...


In [ ]:
# Step 6: Generate both difficulty levels with image/formula awareness

def build_image_context(preprocessed_pages: list) -> str:
    """Builds a reference string listing what images exist per page."""
    lines = []
    for p in preprocessed_pages:
        if p["embedded_images"] or p["display_formulas"]:
            lines.append(
                f"Page {p['page_num']}: "
                f"{len(p['embedded_images'])} figure(s), "
                f"{len(p['display_formulas'])} block formula(s) "
                f"[e.g. {p['display_formulas'][:1]}]"
            )
    return "\n".join(lines)

image_context = build_image_context(preprocessed_pages)

# ── EASY Material ─────────────────────────────────────────────────────────────
EASY_PROMPT = f"""
You are a science communicator writing for high-school students with no prior physics background.

Master content:
{master_content}

Available figures and formulas from the document:
{image_context}

Create an EASY learning guide with these rules:
1. Use simple, everyday language and analogies (e.g. compare electromagnetic waves to water ripples)
2. Structure: Introduction → Key Ideas (one per section) → Simple Visual Description → Fun Facts → Quick Recap
3. For each formula encountered: show it, then explain it in plain English (e.g. "c = speed of light, about 3 lakh km/s!")
4. Reference figures where helpful: write [Figure from Page X] as a placeholder
5. Use bullet points and short paragraphs
6. End with a 5-question beginner quiz
7. NO heavy derivations. Focus on intuition and real-world applications.

Format: Markdown with clear H1/H2/H3 headings.
"""

HARD_PROMPT = f"""
You are a university physics professor writing for students who have completed introductory EM theory.

Master content:
{master_content}

Available figures and formulas from the document:
{image_context}

Create an ADVANCED/DIFFICULT learning guide with these rules:
1. Use precise technical language, standard physics notation
2. Structure: Theoretical Motivation → Maxwell's Equations → Derivations → Wave Properties → Spectrum Analysis → Problem Set
3. Include ALL key equations in LaTeX ($$...$$), show how they are derived or connected
4. Reference figures where helpful: write [Figure from Page X, original diagram]
5. Discuss displacement current rigorously: why it was needed, its mathematical form
6. Include quantitative examples from the text (e.g. Example 8.1, 8.2 style problems)
7. End with 5 challenging problems requiring mathematical working
8. Discuss implications: refractive index, energy density, Poynting vector if inferrable

Format: Markdown with clear H1/H2/H3 headings, LaTeX for all equations.
"""

def generate_material(prompt: str, label: str) -> str:
    print(f"\n🧠 Generating {label} material...")

    # Primary: Gemini
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        resp = model.generate_content(prompt)
        text = resp.text.strip()
        print(f"  ✅ {label} generated via Gemini ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"  [Gemini failed]: {e}")

    # Secondary: Gemma via OpenRouter
    try:
        headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
        payload = {"model": "google/gemma-3-27b-it:free",
                   "messages": [{"role": "user", "content": prompt[:12000]}]}
        resp = requests.post("https://openrouter.ai/api/v1/chat/completions",
                             headers=headers, json=payload, timeout=120)
        text = resp.json()["choices"][0]["message"]["content"].strip()
        print(f"  ✅ {label} generated via Gemma ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"  [Gemma failed]: {e}")

    return f"# {label}\n\n⚠️ Generation failed. Please retry."

easy_material  = generate_material(EASY_PROMPT,  "EASY")
hard_material  = generate_material(HARD_PROMPT,   "DIFFICULT")

# Save raw markdown
with open("easy_material.md",  "w") as f: f.write(easy_material)
with open("hard_material.md",  "w") as f: f.write(hard_material)
print("\n✅ Both materials saved as .md files")


🧠 Generating EASY material...
  [Gemini failed]: name 'genai' is not defined
  [Gemma failed]: 'choices'

🧠 Generating DIFFICULT material...
  [Gemini failed]: name 'genai' is not defined
  [Gemma failed]: 'choices'

✅ Both materials saved as .md files


In [ ]:
# Step 7: Export both materials to styled HTML with MathJax rendering + images

import base64
from pathlib import Path

def image_to_data_uri(img_path: str) -> str:
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = Path(img_path).suffix.lstrip(".")
    return f"data:image/{ext};base64,{b64}"

def build_image_gallery_html(preprocessed_pages: list) -> str:
    """Injects actual page images as a reference gallery at the end."""
    html = '<div class="image-gallery"><h2>📸 Original Document Images</h2>'
    for p in preprocessed_pages:
        if p["embedded_images"]:
            html += f'<h4>Page {p["page_num"]} — Figures</h4>'
            for img_path in p["embedded_images"]:
                if Path(img_path).exists():
                    uri = image_to_data_uri(img_path)
                    html += f'<img src="{uri}" alt="Figure from page {p["page_num"]}" class="doc-img"/>'
    html += '</div>'
    return html

def markdown_to_html_body(md_text: str) -> str:
    """Very lightweight md→html (install markdown lib for production)."""
    try:
        import markdown
        return markdown.markdown(md_text, extensions=["fenced_code", "tables"])
    except ImportError:
        !pip install -q markdown
        import markdown
        return markdown.markdown(md_text, extensions=["fenced_code", "tables"])

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title}</title>
<script>
  MathJax = {{
    tex: {{ inlineMath: [['$','$'], ['\\\\(','\\\\)']], displayMath: [['$$','$$']] }},
    svg: {{ fontCache: 'global' }}
  }};
</script>
<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"></script>
<style>
  :root {{ --bg: {bg}; --accent: {accent}; --text: #1a1a2e; }}
  body {{ font-family: 'Segoe UI', sans-serif; background: var(--bg);
          color: var(--text); max-width: 900px; margin: 0 auto; padding: 2rem; line-height: 1.7; }}
  h1 {{ color: var(--accent); border-bottom: 3px solid var(--accent); padding-bottom: .5rem; }}
  h2 {{ color: var(--accent); margin-top: 2rem; }}
  h3 {{ color: #555; }}
  code {{ background: #f0f0f0; padding: 2px 6px; border-radius: 4px; }}
  pre  {{ background: #1e1e1e; color: #d4d4d4; padding: 1rem; border-radius: 8px; overflow-x: auto; }}
  blockquote {{ border-left: 4px solid var(--accent); padding-left: 1rem; color: #555; font-style: italic; }}
  .badge {{ display: inline-block; background: var(--accent); color: white;
             padding: 4px 12px; border-radius: 20px; font-size: .85rem; margin-bottom: 1rem; }}
  .doc-img {{ max-width: 100%; border: 1px solid #ddd; border-radius: 8px; margin: 8px 0; }}
  .image-gallery {{ background: #f9f9f9; padding: 1rem; border-radius: 8px; margin-top: 3rem; }}
  table {{ border-collapse: collapse; width: 100%; }}
  th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
  th {{ background: var(--accent); color: white; }}
</style>
</head>
<body>
<div class="badge">{badge}</div>
<h1>{title}</h1>
{body}
{gallery}
</body>
</html>"""

def export_html(md_text: str, title: str, filename: str,
                bg: str, accent: str, badge: str,
                preprocessed_pages: list, include_gallery: bool = True):
    body = markdown_to_html_body(md_text)
    gallery = build_image_gallery_html(preprocessed_pages) if include_gallery else ""
    html = HTML_TEMPLATE.format(
        title=title, bg=bg, accent=accent, badge=badge,
        body=body, gallery=gallery
    )
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Exported → {filename}")

export_html(
    easy_material,
    title="Electromagnetic Waves — Beginner's Guide 🌊",
    filename="easy_material.html",
    bg="#fffde7", accent="#f57f17",
    badge="🟢 EASY — For Beginners",
    preprocessed_pages=preprocessed_pages,
    include_gallery=True      # show simplified figures
)

export_html(
    hard_material,
    title="Electromagnetic Waves — Advanced Technical Guide",
    filename="hard_material.html",
    bg="#e8eaf6", accent="#283593",
    badge="🔴 DIFFICULT — For Advanced Learners",
    preprocessed_pages=preprocessed_pages,
    include_gallery=True      # full figures + equations
)

✅ Exported → easy_material.html
✅ Exported → hard_material.html


In [ ]:
# Step 8: Zip everything for download / GitHub upload
import shutil, zipfile

output_files = [
    "easy_material.md", "easy_material.html",
    "hard_material.md", "hard_material.html",
    "structured_doc.json"
]

with zipfile.ZipFile("task2_output.zip", "w") as zf:
    for f in output_files:
        if Path(f).exists():
            zf.write(f)
    # Add page images folder
    for p in preprocessed_pages:
        for img in p["embedded_images"]:
            if Path(img).exists():
                zf.write(img)

print("✅ All outputs zipped → task2_output.zip")

# Download in Colab
from google.colab import files
files.download("task2_output.zip")
files.download("easy_material.html")
files.download("hard_material.html")

✅ All outputs zipped → task2_output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Model Changed

In [ ]:
# Step 2: OCR — one image at a time, raw REST API (no genai SDK), 30s timeout, saves after each page

import pytesseract
import base64, requests, os, json, time
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

# ── CONFIG ────────────────────────────────────────────────────────────────────
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

TIMEOUT_SECS       = 30                          # hard timeout per model attempt
MIN_TEXT_LEN       = 30                          # minimum chars to accept a result
RESULTS_FILE       = "ocr_results.json"          # incremental save path

OCR_PROMPT = (
    "You are an expert OCR engine. Extract ALL text from this image EXACTLY as it appears. "
    "Rules: "
    "Math equations → LaTeX: inline $...$, block $$...$$. "
    "Preserve section headings, figure captions, table content. "
    "Do NOT summarize or skip anything. "
    "Output raw extracted text only."
)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def call_with_timeout(fn, timeout: int = TIMEOUT_SECS):
    """
    Runs fn() with a hard timeout.
    Returns (result, elapsed, timed_out).
    """
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(fn)
        try:
            result  = future.result(timeout=timeout)
            elapsed = round(time.time() - t0, 1)
            return result, elapsed, False
        except FuturesTimeout:
            return None, timeout, True
        except Exception as e:
            return None, round(time.time() - t0, 1), False

# def load_existing_results() -> dict:
#     """Load already-processed pages so we can resume if interrupted."""
#     if Path(RESULTS_FILE).exists():
#         with open(RESULTS_FILE) as f:
#             data = json.load(f)
#         print(f"♻️  Resuming — {len(data)} pages already processed")
#         return data            # {str(page_num): result_dict}
#     return {}

def save_result(page_num: int, result: dict, all_results: dict):
    """Save immediately after each page — crash-safe."""
    all_results[str(page_num)] = result
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

# ── MODEL 1: Gemini via raw REST API ─────────────────────────────────────────
def _gemini_ocr(image_path: str) -> str:
    """
    Calls Gemini 3.1 Flash Lite Preview directly via REST — no SDK dependency.
    Raises on any error so call_with_timeout can catch it.
    """
    b64  = image_to_base64(image_path)
    url  = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"gemini-3.1-flash-lite-preview:generateContent?key={GEMINI_API_KEY}"
    )
    payload = {
        "contents": [{
            "parts": [
                {"text": OCR_PROMPT},
                {
                    "inline_data": {
                        "mime_type": "image/png",
                        "data": b64
                    }
                }
            ]
        }],
        "generationConfig": {
            "temperature": 0             # deterministic — best for OCR
        }
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECS)
    resp.raise_for_status()
    data = resp.json()
    return data["candidates"][0]["content"]["parts"][0]["text"].strip()


# ── MODEL 2: Qwen VL via OpenRouter REST API ──────────────────────────────────
def _qwen_ocr(image_path: str) -> str:
    """
    Calls Qwen 2.5 VL via OpenRouter REST.
    Raises on any error so call_with_timeout can catch it.
    """
    b64  = image_to_base64(image_path)
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://colab.research.google.com",   # required by OpenRouter
        },
        json={
            "model": "qwen/qwen3.6-plus-preview",
            "messages": [{
                "role": "user",
                "content": [
                    {
                        "type":      "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"}
                    },
                    {
                        "type": "text",
                        "text": OCR_PROMPT
                    }
                ]
            }],
            "temperature": 0
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()


# ── MODEL 3: Tesseract (local fallback — always fast) ─────────────────────────
def _tesseract_ocr(image_path: str) -> str:
    """
    Local Tesseract — no network, almost never times out.
    Raises on any error so call_with_timeout can catch it.
    """
    img  = Image.open(image_path)
    text = pytesseract.image_to_string(img, config="--psm 6 --oem 3")
    if not text.strip():
        raise ValueError("Tesseract returned empty text")
    return text.strip()


# ── PER-PAGE OCR WITH FALLBACK CHAIN ─────────────────────────────────────────
MODELS = [
    ("Gemini 3.1 Flash Lite Preview (REST)", _gemini_ocr),
    ("Qwen 3.6 Plus Preview (OpenRouter)", _qwen_ocr),
    ("Tesseract (local)",        _tesseract_ocr),
]

def ocr_one_page(page: dict) -> dict:
    """
    Tries each model in order with TIMEOUT_SECS timeout.
    Returns result dict as soon as one succeeds.
    Saves to disk immediately after success or total failure.
    """
    pn         = page["page_num"]
    image_path = page["image_path"]

    print(f"\n{'─'*50}")
    print(f"📄 Page {pn}  →  {Path(image_path).name}")

    for model_name, model_fn in MODELS:
        print(f"  ↳ [{model_name}] calling...", end=" ", flush=True)

        text, elapsed, timed_out = call_with_timeout(
            lambda fn=model_fn: fn(image_path),   # lambda captures current fn
            timeout=TIMEOUT_SECS
        )

        if timed_out:
            print(f"⏱️  TIMED OUT after {TIMEOUT_SECS}s — next model")
            continue

        if text and len(text.strip()) >= MIN_TEXT_LEN:
            print(f"✅ {elapsed}s — {len(text)} chars extracted")
            return {
                "page_num":        pn,
                "text":            text.strip(),
                "source":          model_name,
                "elapsed_sec":     elapsed,
                "image_path":      image_path,
                "embedded_images": page["embedded_images"],
            }

        print(f"⚠️  {elapsed}s — too little text ({len(text or '')} chars) — next model")

    # All models failed
    print(f"  ❌ All models failed for page {pn}")
    return {
        "page_num":        pn,
        "text":            "",
        "source":          "FAILED",
        "elapsed_sec":     0,
        "image_path":      image_path,
        "embedded_images": page["embedded_images"],
    }


# ── MAIN LOOP: one page at a time, saves after each ──────────────────────────
def run_ocr_sequential(pages_data: list) -> list[dict]:
    import time
    t_start     = time.time()
    all_results = load_existing_results()          # resume support
    ocr_results = []

    for page in pages_data:
        pn = page["page_num"]

        # Skip if already processed (resume mode)
        # if str(pn) in all_results:
        #     print(f"  ⏭️  Page {pn} already done — skipping")
        #     ocr_results.append(all_results[str(pn)])
        #     continue

        result = ocr_one_page(page)

        save_result(pn, result, all_results)       # 💾 save immediately
        ocr_results.append(result)

    # ── Final summary ─────────────────────────────────────────────────────────
    total_time = round(time.time() - t_start, 1)
    success    = sum(1 for r in ocr_results if r["text"])
    sources    = {}
    for r in ocr_results:
        sources[r["source"]] = sources.get(r["source"], 0) + 1

    print(f"\n{'='*55}")
    print(f"✅  OCR complete in {total_time}s")
    print(f"    Extracted : {success} / {len(ocr_results)} pages")
    print(f"    Failed    : {len(ocr_results) - success} pages")
    print(f"    Results   : saved to '{RESULTS_FILE}'")
    print("    Sources   :")
    for src, count in sources.items():
        print(f"      {src}: {count} page(s)")
    print(f"{'='*55}")

    return sorted(ocr_results, key=lambda x: x["page_num"])


# ── RUN ───────────────────────────────────────────────────────────────────────
ocr_results = run_ocr_sequential(pages_data)

♻️  Resuming — 14 pages already processed

──────────────────────────────────────────────────
📄 Page 1  →  page_001.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 3.7s — 1497 chars extracted

──────────────────────────────────────────────────
📄 Page 2  →  page_002.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 5.9s — 3019 chars extracted

──────────────────────────────────────────────────
📄 Page 3  →  page_003.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 6.8s — 3872 chars extracted

──────────────────────────────────────────────────
📄 Page 4  →  page_004.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 5.8s — 3080 chars extracted

──────────────────────────────────────────────────
📄 Page 5  →  page_005.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 5.5s — 2704 chars extracted

──────────────────────────────────────────────────
📄 Page 6  →  page_006.png
  ↳ [Gemini 3.1 Flash Lite Preview (REST)] calling... ✅ 5.5s — 2977 ch

In [ ]:
# Step 4: LLM structures raw text into sections — raw REST API only, no genai SDK

import json, requests, re, time, os
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")          # ← add this to Colab secrets

TIMEOUT_SECS       = 45
STRUCTURED_FILE    = "structured_doc.json"

# ── PROMPT ────────────────────────────────────────────────────────────────────
SECTION_PROMPT_TEMPLATE = """You are a document structuring assistant. Given the raw OCR text below (extracted from a Physics textbook chapter), organize it into a clean structured JSON.

Output ONLY valid JSON. No markdown fences, no explanation, no preamble.

JSON schema:
{{
  "chapter_title": "string",
  "sections": [
    {{
      "section_id": "string (e.g. 8.1, 8.2.1)",
      "title": "string",
      "page_range": [start_page, end_page],
      "content": "string (cleaned text for this section)",
      "key_formulas": ["LaTeX formula strings"],
      "key_concepts": ["concept1", "concept2"],
      "has_figures": true,
      "difficulty_hint": "foundational | intermediate | advanced"
    }}
  ],
  "summary_points": ["point1", "point2"]
}}

Raw OCR text (pages {start_page}–{end_page}):
{text}"""


# ── HELPERS ───────────────────────────────────────────────────────────────────
def call_with_timeout(fn, timeout: int = TIMEOUT_SECS):
    """Runs fn() with a hard timeout. Returns (result, elapsed, timed_out)."""
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(fn)
        try:
            result  = future.result(timeout=timeout)
            return result, round(time.time() - t0, 1), False
        except FuturesTimeout:
            return None, timeout, True
        except Exception as e:
            print(f"    [call error]: {e}")
            return None, round(time.time() - t0, 1), False


def clean_json_string(raw: str) -> str:
    """Strips markdown fences and finds the outermost { } block."""
    raw   = re.sub(r"```(?:json)?", "", raw).strip("`").strip()
    start = raw.find("{")
    end   = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        return raw[start:end + 1]
    return raw


def parse_json_safe(raw: str) -> dict | None:
    try:
        return json.loads(clean_json_string(raw))
    except json.JSONDecodeError as e:
        print(f"    [JSON parse error]: {e}")
        return None


# ── MODEL 1: Gemini 3.0 Flash Preview via REST ───────────────────────────────────────
def _gemini_structure(prompt: str) -> dict | None:
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"gemini-3-flash-preview:generateContent?key={GEMINI_API_KEY}"
    )
    resp = requests.post(
        url,
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature":      0.1,
                "responseMimeType": "application/json"    # forces pure JSON output
            }
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    raw = resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    return parse_json_safe(raw)


# ── MODEL 2: Gemma 3 27B via OpenRouter REST ──────────────────────────────────
def _openrouter_structure(prompt: str, model_id: str) -> dict | None:
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://colab.research.google.com",
        },
        json={
            "model":           model_id,
            "messages":        [{"role": "user", "content": prompt}],
            "temperature":     0.1,
            "response_format": {"type": "json_object"},
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"].strip()
    return parse_json_safe(raw)


# ── MODEL 3: GPT OSS 120B via Groq REST ──────────────────────────────────────
# Groq uses an OpenAI-compatible REST API — no SDK needed at all.
# Check your exact model name at: https://console.groq.com/docs/models
GROQ_MODEL_ID = "openai/gpt-oss-120b"   # ← verify from your Groq console

def _groq_structure(prompt: str) -> dict | None:
    """
    Calls Groq's OpenAI-compatible endpoint directly via requests.
    Groq is extremely fast (LPU inference) — usually responds in 2–5s.
    """
    resp = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",        # Groq REST endpoint
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type":  "application/json",
        },
        json={
            "model":       GROQ_MODEL_ID,
            "messages": [
                {
                    "role":    "system",
                    "content": "You are a document structuring assistant. Output ONLY valid JSON, no markdown, no explanation."
                },
                {
                    "role":    "user",
                    "content": prompt
                }
            ],
            "temperature":     0.1,
            "response_format": {"type": "json_object"},            # Groq supports JSON mode
            "max_tokens":      4096,
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"].strip()
    return parse_json_safe(raw)


# ── FALLBACK CHAIN ────────────────────────────────────────────────────────────
STRUCTURE_MODELS = [
    ("Gemini 3.0 Flash Preview (REST)",        lambda p: _gemini_structure(p), 12000),
    ("Gemma 4 31B       (OpenRouter)",  lambda p: _openrouter_structure(p, "google/gemma-4-31b-it:free"), 10000),
    ("GPT OSS 120B      (Groq)", lambda p: _groq_structure(p), 10000),
]
#                                                                                                     ↑ token slice per model


def structure_with_fallback(combined_text: str, start_page: int, end_page: int) -> dict:
    print(f"\n🗂️  Structuring pages {start_page}–{end_page}  ({len(combined_text)} chars)")

    for model_name, model_fn, char_limit in STRUCTURE_MODELS:
        prompt = SECTION_PROMPT_TEMPLATE.format(
            text=combined_text[:char_limit],
            start_page=start_page,
            end_page=end_page
        )

        print(f"  ↳ [{model_name}] calling...", end=" ", flush=True)

        result, elapsed, timed_out = call_with_timeout(
            lambda fn=model_fn, p=prompt: fn(p),
            timeout=TIMEOUT_SECS
        )

        if timed_out:
            print(f"⏱️  TIMED OUT after {TIMEOUT_SECS}s — next model")
            continue

        if result and "sections" in result and len(result["sections"]) > 0:
            print(f"✅ {elapsed}s — {len(result['sections'])} sections found")
            return result

        print(f"⚠️  {elapsed}s — invalid/empty result — next model")

    # ── All models failed → flat fallback ────────────────────────────────────
    print("  ❌ All structuring models failed — using flat fallback")
    return {
        "chapter_title": "Unknown Chapter",
        "sections": [{
            "section_id":      "all",
            "title":           "Full Content",
            "page_range":      [start_page, end_page],
            "content":         combined_text,
            "key_formulas":    [],
            "key_concepts":    [],
            "has_figures":     False,
            "difficulty_hint": "intermediate"
        }],
        "summary_points": []
    }


# ── BUILD COMBINED TEXT FROM PREPROCESSED PAGES ───────────────────────────────
full_text = "\n\n".join(
    f"[PAGE {p['page_num']}]\n{p['cleaned_text']}"
    for p in preprocessed_pages
    if p.get("cleaned_text", "").strip()
)

start_page = preprocessed_pages[0]["page_num"]
end_page   = preprocessed_pages[-1]["page_num"]

print(f"📝 Combined text: {len(full_text)} chars across pages {start_page}–{end_page}")

# ── RUN ───────────────────────────────────────────────────────────────────────
structured_doc = structure_with_fallback(full_text, start_page, end_page)

# ── SAVE ──────────────────────────────────────────────────────────────────────
with open(STRUCTURED_FILE, "w", encoding="utf-8") as f:
    json.dump(structured_doc, f, indent=2, ensure_ascii=False)

print(f"\n✅ Structured JSON saved → {STRUCTURED_FILE}")
print(f"   Chapter : {structured_doc['chapter_title']}")
print(f"   Sections: {len(structured_doc['sections'])}")
print(f"   Summary : {len(structured_doc['summary_points'])} points")
print("\nPreview:")
print(json.dumps(structured_doc, indent=2)[:800], "\n...")

📝 Combined text: 37803 chars across pages 1–14

🗂️  Structuring pages 1–14  (37803 chars)
  ↳ [Gemini 3.0 Flash Preview (REST)] calling... ✅ 9.4s — 2 sections found

✅ Structured JSON saved → structured_doc.json
   Chapter : ELECTROMAGNETIC WAVES
   Sections: 2
   Summary : 5 points

Preview:
{
  "chapter_title": "ELECTROMAGNETIC WAVES",
  "sections": [
    {
      "section_id": "8.1",
      "title": "INTRODUCTION",
      "page_range": [
        1,
        2
      ],
      "content": "In Chapter 4, we learnt that an electric current produces magnetic field and that two current-carrying wires exert a magnetic force on each other. Further, in Chapter 6, we have seen that a magnetic field changing with time gives rise to an electric field. James Clerk Maxwell argued that not only an electric current but also a time-varying electric field generates magnetic field. Maxwell formulated a set of equations involving electric and magnetic fields, and their sources, the charge and current densiti

In [ ]:
# Step 5: Build master document using LangChain Refine chain
!pip install -q langchain-google-genai langchain-core langchain-classic

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# Ensure the API key is retrieved from environment
API_KEY = os.environ.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=API_KEY,
    temperature=0.3
)

# Convert structured sections into LangChain Documents
lc_docs = []
for section in structured_doc["sections"]:
    content = (
        f"SECTION: {section['section_id']} — {section['title']}\n"
        f"Pages: {section['page_range']}\n"
        f"Difficulty: {section['difficulty_hint']}\n\n"
        f"{section['content']}\n\n"
        f"Key Formulas: {'; '.join(section['key_formulas'])}\n"
        f"Key Concepts: {', '.join(section['key_concepts'])}"
    )
    lc_docs.append(Document(page_content=content, metadata={
        "section_id": section["section_id"],
        "title": section["title"],
        "page_range": section["page_range"],
        "has_figures": section["has_figures"],
    }))

print(f"📄 {len(lc_docs)} LangChain Documents created")

# Refine chain — iteratively builds a coherent understanding
refine_initial_prompt = PromptTemplate(
    input_variables=["text"],
    template=(
        "You are a physics expert. Read this section and extract a clear, complete understanding:\n\n"
        "{text}\n\n"
        "Write a comprehensive summary preserving all key equations, concepts, and relationships."
    )
)
refine_iter_prompt = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=(
        "Here is the understanding built so far:\n{existing_answer}\n\n"
        "Now refine and extend it with this new section:\n{text}\n\n"
        "Output a single comprehensive document. Preserve all formulas and key concepts."
    )
)

refine_chain = load_summarize_chain(
    llm,
    chain_type="refine",
    question_prompt=refine_initial_prompt,
    refine_prompt=refine_iter_prompt,
    verbose=False
)

print("⚙️ Running Refine chain (this may take ~1–2 min)...")
refined_output = refine_chain.invoke(lc_docs)
master_content = refined_output["output_text"]

print(f"\n✅ Master document built ({len(master_content)} chars)")
print(master_content[:500], "...")

📄 2 LangChain Documents created
⚙️ Running Refine chain (this may take ~1–2 min)...

✅ Master document built (4536 chars)
This comprehensive document synthesizes the theoretical foundations of electromagnetic theory, focusing on Maxwell’s unification and the pivotal discovery of the displacement current.

---

# The Foundations of Electromagnetic Waves

This theory establishes the unified framework of electromagnetism by synthesizing the work of Oersted, Ampere, and Faraday into a single, consistent set of laws developed by James Clerk Maxwell.

## 1. The Context of Unification
The development of classical electrom ...


In [ ]:
# Step 6 (UPGRADED): Generate both difficulty levels — goal-aligned prompts
# Goal: clear communication + content structuring + audience adaptation

import requests, os, time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")

TIMEOUT_SECS  = 90
GROQ_MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"  # verify at console.groq.com

# ── TIMEOUT WRAPPER ────────────────────────────────────────────────────────────
def call_with_timeout(fn, timeout: int = TIMEOUT_SECS):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(fn)
        try:
            result  = future.result(timeout=timeout)
            return result, round(time.time() - t0, 1), False
        except FuturesTimeout:
            return None, timeout, True
        except Exception as e:
            print(f"    [call error]: {e}")
            return None, round(time.time() - t0, 1), False


# ── IMAGE / FORMULA CONTEXT BUILDER ───────────────────────────────────────────
def build_image_context(preprocessed_pages: list) -> str:
    lines = []
    for p in preprocessed_pages:
        if p.get("embedded_images") or p.get("display_formulas"):
            lines.append(
                f"Page {p['page_num']}: "
                f"{len(p.get('embedded_images', []))} figure(s), "
                f"{len(p.get('display_formulas', []))} block formula(s) "
                f"[e.g. {p.get('display_formulas', [])[:1]}]"
            )
    return "\n".join(lines) if lines else "No embedded figures detected."


# ══════════════════════════════════════════════════════════════════════════════
# EASY PROMPT — designed for ZERO prior knowledge readers
# Focus: intuition, plain English, zero assumptions, real-world grounding
# ══════════════════════════════════════════════════════════════════════════════
def build_easy_prompt(master_content: str, image_context: str) -> str:
    return f"""You are writing a learning guide for a 16-year-old who has never studied physics beyond basic school science.
Your single most important job is: every reader finishes understanding the topic with ZERO confusion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MASTER CONTENT (extracted from textbook):
{master_content}

FIGURES AND FORMULAS AVAILABLE:
{image_context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DOCUMENT STRUCTURE — follow this EXACTLY:

# [Chapter Title] — Beginner's Guide 🌊

## 📋 What You Will Learn
- List 4–5 outcomes in plain language (e.g. "You will understand WHY light is a wave")

## 🗺️ Quick Roadmap
One-paragraph overview of the whole topic in the simplest possible language.

## [For EACH main section of the textbook, create:]
### [Section Name in Plain English — never use jargon as a heading]
**The Big Idea** (1 sentence, no jargon)
**Real-World Analogy**: Compare the concept to something from daily life
**Explanation**: 2–4 short paragraphs, max 2 sentences each
**The Formula (if any)**: Show it, then immediately explain every symbol in plain English
  → Example format: $c = 3 \\times 10^8$ m/s means "light travels 30 crore metres every second"
  → NEVER show a formula without an immediate plain-English explanation
**[Figure from Page X]** — describe what the figure shows in 1 simple sentence
**Key Takeaway** (one bold sentence)

## 🌍 Real-World Applications
For each type of EM wave: one concrete real-world application in 2 sentences

## 💡 5 Fun Facts
Surprising, memorable facts from the content

## 🔄 Quick Recap
Bullet summary — one line per main idea, no jargon

## ✅ Test Yourself (Beginner Quiz)
5 questions — multiple choice, answers provided at the end
Questions must test concepts, NOT formula recall

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT COMMUNICATION RULES:
1. NEVER use a technical term without defining it in the same sentence
   BAD:  "The displacement current resolves the inconsistency."
   GOOD: "The displacement current (a type of 'pretend' current Maxwell invented) fixes a gap in the old rule."
2. Sentences must be SHORT — max 20 words each
3. NO LaTeX display blocks ($$...$$) — inline $...$ only, always followed by plain English
4. NO phrases: "it can be shown", "trivially", "clearly", "obviously"
5. Use active voice: "Maxwell discovered" not "it was discovered by Maxwell"
6. Every heading must make sense without reading the content below it
7. If you reference a figure, describe what it shows — don't just say "see Figure 8.1"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Format: Markdown. Target length: 1200–1800 words."""


# ══════════════════════════════════════════════════════════════════════════════
# HARD PROMPT — designed for university-level readers with EM theory background
# Focus: rigorous derivation, mathematical depth, conceptual precision
# ══════════════════════════════════════════════════════════════════════════════
def build_hard_prompt(master_content: str, image_context: str) -> str:
    return f"""You are a university physics professor authoring a technical study guide for
second-year undergraduates who have completed: vector calculus, electrostatics, magnetostatics,
and Faraday's law. They have NOT yet studied EM waves or Maxwell's full equations.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MASTER CONTENT (extracted from textbook):
{master_content}

FIGURES AND FORMULAS AVAILABLE:
{image_context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DOCUMENT STRUCTURE — follow this EXACTLY:

# [Chapter Title] — Advanced Technical Guide

## 📌 Prerequisites
List exactly what prior knowledge is assumed (vector calculus, Gauss's law, etc.)

## 🎯 Learning Objectives
5–6 precise, measurable outcomes (e.g. "Derive the wave equation from Maxwell's equations")

## 1. Theoretical Motivation
- Why was Ampere's circuital law incomplete?
- Show the mathematical inconsistency with a capacitor charging scenario
- Introduce the need for displacement current with full mathematical argument

## 2. Displacement Current — Rigorous Treatment
- Define: $$i_d = \\varepsilon_0 \\frac{{d\\Phi_E}}{{dt}}$$
- Derive why this term is needed for self-consistency
- Generalised Ampere-Maxwell law — full form with both terms
- Physical interpretation: what displacement current IS and IS NOT
- [Figure from Page X] — reference and explain each relevant diagram precisely

## 3. Maxwell's Equations in Vacuum
Present all 4 equations in integral form with:
- Equation in LaTeX ($$...$$)
- Physical law it encodes
- What symmetry it reveals

## 4. Derivation of Electromagnetic Waves
- From Maxwell's equations → wave equation step by step
- Show that $c = 1/\\sqrt{{\\mu_0 \\varepsilon_0}}$ emerges naturally
- Prove E and B are perpendicular to each other AND to propagation direction
- Relation between amplitudes: $B_0 = E_0/c$
- Energy transport: mention Poynting vector $\\vec{{S}} = \\frac{{1}}{{\\mu_0}}\\vec{{E}} \\times \\vec{{B}}$

## 5. Wave Equations and Solutions
- Sinusoidal plane wave solutions with full notation
- Dispersion relation $\\omega = ck$
- Phase velocity in material media: $v = 1/\\sqrt{{\\mu \\varepsilon}}$
- Connection to refractive index

## 6. The Electromagnetic Spectrum
For each spectral region: frequency/wavelength range, production mechanism, detection method, applications
Present as a structured table AND narrative

## 7. Worked Examples
Reproduce and extend each worked example from the source material.
Show full working. State what physical principle each step uses.

## 8. Critical Analysis
- What Maxwell's theory predicts vs. what was experimentally confirmed
- Role of Hertz, Bose, Marconi — historical-theoretical connection
- Limitations of the classical EM wave picture

## 9. Problem Set (Advanced)
5 problems requiring multi-step mathematical working.
Mix of: derivation, numerical calculation, conceptual proof.
Include solutions outline (not full solutions).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT COMMUNICATION RULES:
1. ALL equations in LaTeX: inline $...$ or display $$...$$
2. Define every symbol the FIRST time it appears: e.g. "where $\\mu_0$ is the permeability of free space"
3. Each derivation step must state the physical justification, not just algebra
   BAD:  "Substituting and simplifying gives the wave equation"
   GOOD: "Applying the curl of Faraday's law and using the vector identity ∇×(∇×E),
          then substituting Ampere-Maxwell law, yields the wave equation"
4. Connect sections explicitly: start each section with one sentence linking it to the previous
5. NO informal language — but also NO "it can be shown" or "it is left as exercise"
   without showing the key steps
6. Every figure reference must describe: what the figure shows, what to notice, what it proves
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Format: Markdown with LaTeX. Target length: 2500–3500 words."""


# ── MODEL CALLS (raw REST, no SDK) ─────────────────────────────────────────────

def _gemini_generate(prompt: str) -> str:
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}"
    )
    resp = requests.post(
        url,
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0.3, "maxOutputTokens": 8192}
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    if not text:
        raise ValueError("Empty response from Gemini")
    return text


def _openrouter_generate(prompt: str) -> str:
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://colab.research.google.com",
        },
        json={
            "model":       "google/gemma-3-27b-it:free",
            "messages":    [{"role": "user", "content": prompt[:12000]}],
            "temperature": 0.3,
            "max_tokens":  8192,
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["choices"][0]["message"]["content"].strip()
    if not text:
        raise ValueError("Empty response from Gemma")
    return text


def _groq_generate(prompt: str) -> str:
    resp = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type":  "application/json",
        },
        json={
            "model": GROQ_MODEL_ID,
            "messages": [
                {
                    "role":    "system",
                    "content": (
                        "You are an expert physics educator producing structured learning materials. "
                        "Follow the document structure in the user prompt EXACTLY. "
                        "Use LaTeX for all equations. Produce complete, detailed output."
                    )
                },
                {
                    "role":    "user",
                    "content": prompt[:12000]
                }
            ],
            "temperature": 0.3,
            "max_tokens":  8192,
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["choices"][0]["message"]["content"].strip()
    if not text:
        raise ValueError("Empty response from Groq")
    return text


# ── GENERATION MODELS CHAIN ────────────────────────────────────────────────────
GENERATION_MODELS = [
    ("Gemini 2.0 Flash  (REST)",       _gemini_generate),
    ("Gemma 3 27B       (OpenRouter)", _openrouter_generate),
    ("GPT OSS 120B      (Groq)",       _groq_generate),
]


# ── MAIN GENERATOR WITH FALLBACK + TIMEOUT ────────────────────────────────────
def generate_material(prompt: str, label: str) -> str:
    print(f"\n{'─'*55}")
    print(f"🧠 Generating [{label}] material...")

    for model_name, model_fn in GENERATION_MODELS:
        print(f"  ↳ [{model_name}] calling...", end=" ", flush=True)

        text, elapsed, timed_out = call_with_timeout(
            lambda fn=model_fn, p=prompt: fn(p),
            timeout=TIMEOUT_SECS
        )

        if timed_out:
            print(f"⏱️  TIMED OUT after {TIMEOUT_SECS}s — next model")
            continue

        if text and len(text.strip()) > 300:
            print(f"✅ {elapsed}s — {len(text)} chars generated")
            return text.strip()

        print(f"⚠️  {elapsed}s — too short ({len(text or '')} chars) — next model")

    print(f"  ❌ All models failed for [{label}]")
    return f"# {label}\n\n⚠️ Generation failed across all models. Please retry.\n"


# ── BUILD + RUN ────────────────────────────────────────────────────────────────
image_context = build_image_context(preprocessed_pages)
easy_prompt   = build_easy_prompt(master_content, image_context)
hard_prompt   = build_hard_prompt(master_content, image_context)

print(f"📊 Image context : {len(image_context)} chars")
print(f"📝 Easy prompt   : {len(easy_prompt)} chars")
print(f"📝 Hard prompt   : {len(hard_prompt)} chars")

easy_material = generate_material(easy_prompt,  "EASY")
hard_material = generate_material(hard_prompt,  "DIFFICULT")

# ── SAVE ──────────────────────────────────────────────────────────────────────
with open("easy_material.md", "w", encoding="utf-8") as f:
    f.write(easy_material)
with open("hard_material.md", "w", encoding="utf-8") as f:
    f.write(hard_material)

print(f"\n{'='*55}")
print("✅ Both materials generated and saved:")
print(f"   easy_material.md — {len(easy_material)} chars")
print(f"   hard_material.md — {len(hard_material)} chars")
print(f"{'='*55}")

📊 Image context : 1067 chars
📝 Easy prompt   : 8158 chars
📝 Hard prompt   : 9372 chars

───────────────────────────────────────────────────────
🧠 Generating [EASY] material...
  ↳ [Gemini 2.0 Flash  (REST)] calling... ✅ 11.3s — 7876 chars generated

───────────────────────────────────────────────────────
🧠 Generating [DIFFICULT] material...
  ↳ [Gemini 2.0 Flash  (REST)] calling... ✅ 23.7s — 17740 chars generated

✅ Both materials generated and saved:
   easy_material.md — 7876 chars
   hard_material.md — 17740 chars


In [ ]:
# Step 6: Generate both difficulty levels — raw REST API only, no genai SDK
# Fallback chain: Gemini REST → Gemma (OpenRouter) → GPT OSS 120B (Groq)

import requests, os, time, json
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")

TIMEOUT_SECS       = 90          # generation needs more time than OCR/structuring
GROQ_MODEL_ID      = "qwen/qwen3-32b"  # ← verify from console.groq.com/docs/models

# ── TIMEOUT WRAPPER (same pattern as previous steps) ─────────────────────────
def call_with_timeout(fn, timeout: int = TIMEOUT_SECS):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(fn)
        try:
            result  = future.result(timeout=timeout)
            return result, round(time.time() - t0, 1), False
        except FuturesTimeout:
            return None, timeout, True
        except Exception as e:
            print(f"    [call error]: {e}")
            return None, round(time.time() - t0, 1), False


# ── IMAGE / FORMULA CONTEXT BUILDER ──────────────────────────────────────────
def build_image_context(preprocessed_pages: list) -> str:
    lines = []
    for p in preprocessed_pages:
        if p.get("embedded_images") or p.get("display_formulas"):
            lines.append(
                f"Page {p['page_num']}: "
                f"{len(p.get('embedded_images', []))} figure(s), "
                f"{len(p.get('display_formulas', []))} block formula(s) "
                f"[e.g. {p.get('display_formulas', [])[:1]}]"
            )
    return "\n".join(lines) if lines else "No embedded figures detected."


# ── PROMPTS ───────────────────────────────────────────────────────────────────
def build_easy_prompt(master_content: str, image_context: str) -> str:
    return f"""You are a science communicator writing for high-school students with no prior physics background.

Master content:
{master_content}

Available figures and formulas from the document:
{image_context}

Create an EASY learning guide with these rules:
1. Use simple, everyday language and analogies (e.g. compare electromagnetic waves to water ripples)
2. Structure: Introduction → Key Ideas (one per section) → Simple Visual Description → Fun Facts → Quick Recap
3. For each formula encountered: show it, then explain it in plain English (e.g. "c = speed of light, about 3 lakh km/s!")
4. Reference figures where helpful: write [Figure from Page X] as a placeholder
5. Use bullet points and short paragraphs
6. End with a 5-question beginner quiz
7. NO heavy derivations. Focus on intuition and real-world applications.

Format: Markdown with clear H1/H2/H3 headings."""


def build_hard_prompt(master_content: str, image_context: str) -> str:
    return f"""You are a university physics professor writing for students who have completed introductory EM theory.

Master content:
{master_content}

Available figures and formulas from the document:
{image_context}

Create an ADVANCED/DIFFICULT learning guide with these rules:
1. Use precise technical language, standard physics notation
2. Structure: Theoretical Motivation → Maxwell's Equations → Derivations → Wave Properties → Spectrum Analysis → Problem Set
3. Include ALL key equations in LaTeX ($$...$$), show how they are derived or connected
4. Reference figures where helpful: write [Figure from Page X, original diagram]
5. Discuss displacement current rigorously: why it was needed, its mathematical form
6. Include quantitative examples from the text (e.g. Example 8.1, 8.2 style problems)
7. End with 5 challenging problems requiring mathematical working
8. Discuss implications: refractive index, energy density, Poynting vector if inferrable

Format: Markdown with clear H1/H2/H3 headings, LaTeX for all equations."""


# ── MODEL 1: Gemini 3.1 Pro via REST ───────────────────────────────────────
def _gemini_generate(prompt: str) -> str:
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"gemini-3.1-pro-preview:generateContent?key={GEMINI_API_KEY}"
    )
    resp = requests.post(
        url,
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": 0.4,       # slight creativity for teaching material
                "maxOutputTokens": 65536
            }
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    if not text:
        raise ValueError("Empty response from Gemini")
    return text


# ── MODEL 2: Gemma 4 31B via OpenRouter REST ──────────────────────────────────
def _openrouter_generate(prompt: str) -> str:
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://colab.research.google.com",
        },
        json={
            "model":      "google/gemma-4-31b-it:free",
            "messages":   [{"role": "user", "content": prompt[:]}],
            "temperature": 0.4,
            "max_tokens":  65536,
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["choices"][0]["message"]["content"].strip()
    if not text:
        raise ValueError("Empty response from Gemma")
    return text


# ── MODEL 3: GPT OSS 120B via Groq REST ──────────────────────────────────────
def _groq_generate(prompt: str) -> str:
    """
    Groq's OpenAI-compatible endpoint — no SDK, plain requests.
    Groq LPU hardware is very fast, great safety net fallback.
    Check model IDs at: https://console.groq.com/docs/models
    """
    resp = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type":  "application/json",
        },
        json={
            "model": GROQ_MODEL_ID,
            "messages": [
                {
                    "role":    "system",
                    "content": (
                        "You are an expert physics educator. "
                        "Generate detailed, well-structured learning material in Markdown. "
                        "Use LaTeX for all equations."
                    )
                },
                {
                    "role":    "user",
                    "content": prompt[:]      # Groq has large context but be safe
                }
            ],
            "temperature": 0.4,
            "max_tokens":  65536,
        },
        timeout=TIMEOUT_SECS
    )
    resp.raise_for_status()
    text = resp.json()["choices"][0]["message"]["content"].strip()
    if not text:
        raise ValueError("Empty response from Groq")
    return text


# ── GENERATION MODELS CHAIN ───────────────────────────────────────────────────
GENERATION_MODELS = [
    ("Gemini 3.0 Flash (REST)",       _gemini_generate),
    ("Gemma 4 31B       (OpenRouter)", _openrouter_generate),
    (" qwen/qwen3-32b     (Groq)",       _groq_generate),          # ← Groq as 3rd fallback
]


# ── MAIN GENERATOR WITH FALLBACK + TIMEOUT ────────────────────────────────────
def generate_material(prompt: str, label: str) -> str:
    print(f"\n{'─'*55}")
    print(f"🧠 Generating [{label}] material...")

    for model_name, model_fn in GENERATION_MODELS:
        print(f"  ↳ [{model_name}] calling...", end=" ", flush=True)

        text, elapsed, timed_out = call_with_timeout(
            lambda fn=model_fn, p=prompt: fn(p),
            timeout=TIMEOUT_SECS
        )

        if timed_out:
            print(f"⏱️  TIMED OUT after {TIMEOUT_SECS}s — next model")
            continue

        if text and len(text.strip()) > 200:       # sanity: must have real content
            print(f"✅ {elapsed}s — {len(text)} chars generated")
            return text.strip()

        print(f"⚠️  {elapsed}s — response too short ({len(text or '')} chars) — next model")

    # All models failed
    print(f"  ❌ All models failed for [{label}]")
    return f"# {label}\n\n⚠️ Generation failed across all models. Please retry.\n"


# ── BUILD PROMPTS ─────────────────────────────────────────────────────────────
image_context = build_image_context(preprocessed_pages)

easy_prompt   = build_easy_prompt(master_content, image_context)
hard_prompt   = build_hard_prompt(master_content, image_context)

print(f"📊 Image context: {len(image_context)} chars")
print(f"📝 Easy prompt  : {len(easy_prompt)} chars")
print(f"📝 Hard prompt  : {len(hard_prompt)} chars")

# ── GENERATE ──────────────────────────────────────────────────────────────────
easy_material = generate_material(easy_prompt, "EASY")
hard_material = generate_material(hard_prompt, "DIFFICULT")

# ── SAVE ──────────────────────────────────────────────────────────────────────
with open("easy_material.md", "w", encoding="utf-8") as f:
    f.write(easy_material)

with open("hard_material.md", "w", encoding="utf-8") as f:
    f.write(hard_material)

print(f"\n{'='*55}")
print("✅ Both materials saved:")
print(f"   easy_material.md  — {len(easy_material)} chars")
print(f"   hard_material.md  — {len(hard_material)} chars")
print(f"{'='*55}")

📊 Image context: 1067 chars
📝 Easy prompt  : 6436 chars
📝 Hard prompt  : 6592 chars

───────────────────────────────────────────────────────
🧠 Generating [EASY] material...
  ↳ [Gemini 3.0 Flash (REST)] calling... ✅ 27.7s — 6657 chars generated

───────────────────────────────────────────────────────
🧠 Generating [DIFFICULT] material...
  ↳ [Gemini 3.0 Flash (REST)] calling... ✅ 47.7s — 11892 chars generated

✅ Both materials saved:
   easy_material.md  — 6657 chars
   hard_material.md  — 11892 chars


In [ ]:
# Step 7: Export both materials to styled HTML with MathJax rendering + images

import base64
from pathlib import Path

def image_to_data_uri(img_path: str) -> str:
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = Path(img_path).suffix.lstrip(".")
    return f"data:image/{ext};base64,{b64}"

def build_image_gallery_html(preprocessed_pages: list) -> str:
    """Injects actual page images as a reference gallery at the end."""
    html = '<div class="image-gallery"><h2>📸 Original Document Images</h2>'
    for p in preprocessed_pages:
        if p["embedded_images"]:
            html += f'<h4>Page {p["page_num"]} — Figures</h4>'
            for img_path in p["embedded_images"]:
                if Path(img_path).exists():
                    uri = image_to_data_uri(img_path)
                    html += f'<img src="{uri}" alt="Figure from page {p["page_num"]}" class="doc-img"/>'
    html += '</div>'
    return html

def markdown_to_html_body(md_text: str) -> str:
    """Very lightweight md→html (install markdown lib for production)."""
    try:
        import markdown
        return markdown.markdown(md_text, extensions=["fenced_code", "tables"])
    except ImportError:
        !pip install -q markdown
        import markdown
        return markdown.markdown(md_text, extensions=["fenced_code", "tables"])

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title}</title>
<script>
  MathJax = {{
    tex: {{ inlineMath: [['$','$'], ['\\\\(','\\\\)']], displayMath: [['$$','$$']] }},
    svg: {{ fontCache: 'global' }}
  }};
</script>
<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"></script>
<style>
  :root {{ --bg: {bg}; --accent: {accent}; --text: #1a1a2e; }}
  body {{ font-family: 'Segoe UI', sans-serif; background: var(--bg);
          color: var(--text); max-width: 900px; margin: 0 auto; padding: 2rem; line-height: 1.7; }}
  h1 {{ color: var(--accent); border-bottom: 3px solid var(--accent); padding-bottom: .5rem; }}
  h2 {{ color: var(--accent); margin-top: 2rem; }}
  h3 {{ color: #555; }}
  code {{ background: #f0f0f0; padding: 2px 6px; border-radius: 4px; }}
  pre  {{ background: #1e1e1e; color: #d4d4d4; padding: 1rem; border-radius: 8px; overflow-x: auto; }}
  blockquote {{ border-left: 4px solid var(--accent); padding-left: 1rem; color: #555; font-style: italic; }}
  .badge {{ display: inline-block; background: var(--accent); color: white;
             padding: 4px 12px; border-radius: 20px; font-size: .85rem; margin-bottom: 1rem; }}
  .doc-img {{ max-width: 100%; border: 1px solid #ddd; border-radius: 8px; margin: 8px 0; }}
  .image-gallery {{ background: #f9f9f9; padding: 1rem; border-radius: 8px; margin-top: 3rem; }}
  table {{ border-collapse: collapse; width: 100%; }}
  th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
  th {{ background: var(--accent); color: white; }}
</style>
</head>
<body>
<div class="badge">{badge}</div>
<h1>{title}</h1>
{body}
{gallery}
</body>
</html>"""

def export_html(md_text: str, title: str, filename: str,
                bg: str, accent: str, badge: str,
                preprocessed_pages: list, include_gallery: bool = True):
    body = markdown_to_html_body(md_text)
    gallery = build_image_gallery_html(preprocessed_pages) if include_gallery else ""
    html = HTML_TEMPLATE.format(
        title=title, bg=bg, accent=accent, badge=badge,
        body=body, gallery=gallery
    )
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Exported → {filename}")

export_html(
    easy_material,
    title="Electromagnetic Waves — Beginner's Guide 🌊",
    filename="easy_material.html",
    bg="#fffde7", accent="#f57f17",
    badge="🟢 EASY — For Beginners",
    preprocessed_pages=preprocessed_pages,
    include_gallery=True      # show simplified figures
)

export_html(
    hard_material,
    title="Electromagnetic Waves — Advanced Technical Guide",
    filename="hard_material.html",
    bg="#e8eaf6", accent="#283593",
    badge="🔴 DIFFICULT — For Advanced Learners",
    preprocessed_pages=preprocessed_pages,
    include_gallery=True      # full figures + equations
)

✅ Exported → easy_material.html
✅ Exported → hard_material.html


In [40]:
# Step 8: Zip everything for download / GitHub upload
import shutil, zipfile

output_files = [
    "easy_material.md", "easy_material.html",
    "hard_material.md", "hard_material.html",
    "structured_doc.json"
]

with zipfile.ZipFile("task2_output.zip", "w") as zf:
    for f in output_files:
        if Path(f).exists():
            zf.write(f)
    # Add page images folder
    for p in preprocessed_pages:
        for img in p["embedded_images"]:
            if Path(img).exists():
                zf.write(img)

print("✅ All outputs zipped → task2_output.zip")

# Download in Colab
from google.colab import files
files.download("task2_output.zip")
files.download("easy_material.html")
files.download("hard_material.html")

✅ All outputs zipped → task2_output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>